# Taller 3: Simulación Montecarlo - Evaluación de Infraestructura
**Objetivo:** Diseñar y evaluar una librería de simulación estadística aplicada a latencia RAG, pipelines bioinformáticos y tráfico bimodal.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats

# Importaciones desde la librería local según la estructura del repositorio
from mi_simulador.rag_latency import simular_latencia_rag
from mi_simulador.bio_pipeline import importance_sampling_fallos
from mi_simulador.bimodal_traffic import rejection_sampling_bimodal, p_target

# Configuración de estilo
plt.style.use('ggplot')
np.random.seed(42)

## Caso 1: Análisis Estocástico de Latencia en Arquitecturas RAG
Se simula el tiempo total de respuesta ($T_{total}$) compuesto por:
1. **Embeddings:** Distribución Normal ($\mu=10, \sigma=2$).
2. **Búsqueda Vectorial:** Distribución Uniforme ($min=5, max=15$).
3. [cite_start]**Inferencia LLM:** Distribución Lognormal ($\mu=3, \sigma=0.5$) para modelar colas largas[cite: 8, 9, 10].

In [ ]:
n_sim = 100000
latencias = simular_latencia_rag(n_samples=n_sim)

p95 = np.percentile(latencias, 95)
p99 = np.percentile(latencias, 99)

print(f"Percentil 95: {p95:.2f} ms")
print(f"Percentil 99: {p99:.2f} ms")

plt.figure(figsize=(10, 4))
plt.hist(latencias, bins=100, density=True, color='skyblue', alpha=0.7)
plt.axvline(p99, color='red', linestyle='--', label=f'P99 ({p99:.2f}ms)')
plt.title("Distribución de Latencia Total (Caso RAG)")
plt.xlabel("Tiempo (ms)")
plt.legend()
plt.show()

**Argumentación del SLA:**
Definimos un SLA de **$T = 50$ ms**. [cite_start]Según los resultados, el Percentil 99 es aproximadamente `{p99:.2f}` ms[cite: 12]. 
* **Cumplimiento:** Si $T < p99$, la arquitectura **no cumple** el SLA.
* [cite_start]**Propuesta de Optimización:** El componente crítico es la **Inferencia del LLM** debido a su naturaleza Lognormal que genera colas largas[cite: 10, 14]. Se recomienda invertir en hardware de alto rendimiento o técnicas de *quantization* para reducir la varianza en esta etapa.

## Caso 2: Cuellos de Botella en Pipelines Bioinformáticos (Importance Sampling)
Estimamos la probabilidad de que un pipeline supere un límite crítico evento raro. Se utiliza una distribución propuesta $g(x)$ que concentra muestras en la región de fallo para reducir la varianza del estimador.

In [ ]:
limite = 60.0
prob_is, var_is = importance_sampling_fallos(limite_horas=limite, n_samples=100000)

# Comparación conceptual con Monte Carlo Estándar
# En MC Estándar, la varianza sería p(1-p)/n. 
var_mc_estandar = (prob_is * (1 - prob_is)) / 100000

print(f"Probabilidad estimada (IS): {prob_is:.6e}")
print(f"Varianza IS: {var_is:.6e}")
print(f"Varianza MC Estándar (Teórica): {var_mc_estandar:.6e}")
print(f"Factor de reducción de varianza: {var_mc_estandar / var_is:.2f}x")

**Análisis de Eficiencia:**
El método de **Importance Sampling** reduce drásticamente el error computacional comparado con Monte Carlo estándar.
Al desplazar la distribución hacia la zona crítica, obtenemos una estimación precisa de eventos catastróficos con un número de muestras significativamente menor, optimizando el uso de recursos de cómputo.

## Caso 3: Tráfico Sintético Bimodal (Rejection Sampling)
Generación de tráfico que sigue la densidad $p^*(x)$ (picos diurnos y nocturnos).
Se utiliza una envolvente $q(x)$ uniforme y una constante $k$ tal que $k \cdot q(x) \geq p^*(x)$.

In [ ]:
muestras, tasa = rejection_sampling_bimodal(n_samples=10000, k=25)

x = np.linspace(0, 24, 500)
y_teorico = p_target(x)

plt.figure(figsize=(10, 4))
plt.hist(muestras, bins=60, density=True, alpha=0.6, color='green', label='Muestras Aceptadas')
plt.plot(x, y_teorico/np.max(y_teorico) * 0.1, 'r', label='p*(x) escalada') # Ajuste visual
plt.title(f"Tráfico Bimodal - Tasa de Aceptación: {tasa:.2%}")
plt.xlabel("Hora del día")
plt.legend()
plt.show()

**Discusión de Desperdicio de Ciclos:**
La tasa de aceptación de `{tasa:.2%}` indica que por cada muestra aceptada, se rechazaron varias. 
**Impacto de $q(x)$:** Una distribución envolvente muy alejada de la forma de la distribución objetivo genera un alto desperdicio de ciclos de CPU. 
**Optimización:** Si se eligiera una mezcla de normales como $q(x)$ en lugar de una uniforme, la tasa de aceptación subiría, mejorando la eficiencia algorítmica y reduciendo el tiempo de cómputo.